# Level 1 — Knowledge & Retrieval Agent
### Business Demo

---

## What is the Level 1 Agent?

The **Level 1 Knowledge Agent** is the system's information retrieval layer. Think of it as an always-available, instant-access assistant that can answer questions about:

| What you can ask | Where the answer comes from |
|---|---|
| Customer profile, segment, fraud score | Customer database |
| Identity verification status and expiry dates | Customer database |
| Past emails sent to a customer | Email archive |
| CRM agent notes and case history | Agent notes store |
| Internal policies (identity verification, fraud, promotions) | Policy document library |

**No SQL knowledge required.** You ask in plain English — the agent finds the answer.

---

## How it works (in plain terms)

```
Your question
     │
     ▼
Orchestrator classifies intent → routes to Level 1
     │
     ▼
Level 1 decides: customer lookup? Identity Verification check? email search? policy search?
     │
     ▼
Retrieves relevant data from the right source
     │
     ▼
LLM synthesises a clear, cited answer
     │
     ▼
Guardrail checks output for PII leaks or policy violations
     │
     ▼
Answer returned + audit trail logged
```

Every action is logged to an immutable audit trail — who asked, what was retrieved, when.

---

## Setup

Run this cell once to initialise the system. It loads configuration, connects to the database, and builds the agent graph.

In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv
from utils import *

_here = Path(os.getcwd())
_root = _here.parent if _here.name == 'notebooks' else _here
os.chdir(_root); sys.path.insert(0, str(_root)); sys.path.insert(0, str(_root/'notebooks'))
load_dotenv(); os.environ['GUARDRAIL_ENABLED'] = 'false'
print('System Ready | Project root:', _root)


System Ready | Project root: /home/kahloka/projects/agenticAI


---
## Use Case 1 — Customer Profile Lookup

**Business scenario**: A account manager needs a quick overview of a customer before a call — segment, fraud score, Identity Verification status, categories purchased.

The agent retrieves the full profile from the customer database and returns a structured summary. No need to log into multiple systems.

In [2]:
customer_id = "CUST-005"
result = ask("Show me the profile for customer CUST-005", customer_id=customer_id)
show_result(result, title="Customer Profile Query")
customer = result["result"].get("customer", {})
if customer:
    display_customer_profile(customer)
else:
    display_card("Error", "Customer not found", emoji="❌", color="#ef4444")


Category
automotive
clothing
toys


Channel,Consent
marketing,False
email,False
sms,False
phone,False


---
## Use Case 2 — Identity Verification Status

**Business scenario**: Trust & Safety needs to verify whether a customer's Identity Verification is valid before approving a high-value purchase or product application.

The agent returns the current Identity Verification status, expiry date, and how many days remain — or flags if it has already expired.

We'll check **15 customers** with a realistic distribution that reflects the actual customer base: most have **valid** Identity Verification, some have **expired**, and a few are **pending**.

In [3]:
# Get demo customers for identity verification analysis
demo_customers = get_demo_customers()
customers_to_check = demo_customers["identity_verification_mix"]

# Use utility function to display analysis
identity_results = display_identity_status_analysis(customers_to_check)


Customer,Status,Fraud Score
CUST-001,verified,0.820
CUST-004,verified,0.300
CUST-006,verified,0.880
CUST-007,verified,0.000
CUST-008,verified,0.750
CUST-009,verified,0.260
CUST-011,verified,0.840
CUST-012,verified,0.150
CUST-015,verified,0.760
CUST-005,unverified,0.390


---
## Use Case 3 — Email Correspondence Search

**Business scenario**: A customer calls in claiming they never received a Identity Verification renewal reminder. The agent can search the email archive and surface relevant correspondence in seconds.

The agent searches the email collection using semantic similarity — it understands meaning, not just keywords. So *"renewal reminder"* will match emails about *"Identity Verification expiry notice"* even if the exact words differ.

In [4]:
result = ask("Show me emails about Identity Verification renewal reminders sent to customers")

# Use utility function to display results
display_email_search_results(result)


Source,Type,Relevance,Preview
CUST-058-identity_verification_reminder.eml,email,0.786,From: shop@example.com Date: 2025-08-31 Subject: Identity Verification Reminder Dear Lori Williamso...
CUST-060-identity_verification_reminder.eml,email,0.810,From: shop@example.com Date: 2025-05-05 Subject: Identity Verification Reminder Dear Molnárné Simon...
CUST-029-identity_verification_reminder.eml,email,0.860,From: shop@example.com Date: 2026-01-24 Subject: Identity Verification Reminder Dear Jasmine Shepar...
CUST-152-identity_verification_reminder.eml,email,0.866,"From: shop@example.com Date: 2025-08-30 Subject: Identity Verification Reminder Dear Meagan Gamble,..."
CUST-205-identity_verification_reminder.eml,email,0.893,"From: shop@example.com Date: 2026-02-17 Subject: Identity Verification Reminder Dear 张建国, Your ide..."


---
## Use Case 4 — CRM Agent Notes Search

**Business scenario**: A senior manager wants to understand the history of escalations and complaints before a customer review meeting. Instead of manually reading hundreds of CRM notes, the agent surfaces the most relevant ones.

Notes are stored as unstructured text written by frontline agents. The system uses vector search to find semantically relevant notes — even when agents used different wording.

In [5]:
queries = get_test_queries()
result1 = ask(queries["notes_escalations"])
display_search_results(result1, "Escalations & Complaints", emoji="⚠️", color="#ef4444")

result2 = ask(queries["notes_retention"])
display_search_results(result2, "Retention & Loyalty", emoji="🎁", color="#10b981")
chunks2 = result2["result"].get("chunks", [])
if chunks2: display_outcome_analysis(chunks2, "📊 Loyalty Bonus Outcomes")


Source,Type,Relevance,Preview
CUST-030-note.txt,note,0.984,Customer ID: CUST-030 Escalation note (2025-10-17): Customer السيد يسار بارق escalated complaint abo...
CUST-042-note.txt,note,1.035,Customer ID: CUST-042 Escalation note (2025-11-28): Customer السيدة ميار إياد escalated complaint ab...
CUST-206-note.txt,note,1.045,Customer ID: CUST-206 Escalation note (2025-10-30): Customer Δημήτριος Κωνσταντινίδης escalated comp...
CUST-037-note.txt,note,1.057,Customer ID: CUST-037 Escalation note (2026-01-22): Customer صدّام جرهم escalated complaint about in...
CUST-096-note.txt,note,1.058,Customer ID: CUST-096 Escalation note (2025-12-09): Customer الأستاذة حياة الامام escalated complain...


Source,Type,Relevance,Preview
CUST-033-note.txt,note,0.899,Customer ID: CUST-033 Retention note (2026-02-18): Customer showed churn signals. Engagement score: ...
CUST-173-note.txt,note,0.914,Customer ID: CUST-173 Retention note (2026-01-17): Customer showed churn signals. Engagement score: ...
CUST-047-note.txt,note,0.917,Customer ID: CUST-047 Retention note (2026-03-03): Customer showed churn signals. Engagement score: ...
CUST-164-note.txt,note,0.930,Customer ID: CUST-164 Retention note (2026-01-19): Customer showed churn signals. Engagement score: ...
CUST-038-note.txt,note,0.934,Customer ID: CUST-038 Retention note (2026-03-04): Customer showed churn signals. Engagement score: ...


---
## Use Case 5 — Policy Document Search (RAG)

**Business scenario**: A product manager needs to quickly check the eligibility rules for the Premium Membership promotion, or a trust & safety officer needs to verify the identity verification policy — without digging through PDF folders.

The agent uses **Retrieval-Augmented Generation (RAG)**:
1. Searches the policy library for relevant sections
2. Passes those sections to the LLM
3. Returns a synthesised, cited answer — not just a raw document dump

The answer always cites the source document so you can verify it.

In [6]:
queries = get_test_queries()
display_policy_qa(queries["policy_questions"], "Policy Document Search (RAG)")


---
## Use Case 6 — Cross-Source Search

**Business scenario**: A manager asks a broad question that spans multiple data sources — policies, emails, and notes. The agent searches all three collections simultaneously and synthesises a unified answer.

This is the power of the multi-collection RAG pipeline: **one question, all sources, one answer**.

In [7]:
queries = get_test_queries()
result = ask(queries["cross_source"])

display_search_results(result, "Cross-Source Answer", emoji="🔍", color="#8b5cf6")

# Show source breakdown
chunks = result["result"].get("chunks", [])
if chunks:
    display_source_breakdown(chunks, "Sources by Collection")


Source,Type,Relevance,Preview
fraud-policy.md,policy,0.851,# Fraud & Risk Management Policy ## Fraud Score Thresholds - 0.0 – 0.4: Low risk — standard process...
CUST-151-we_miss_you.eml,email,1.256,"From: shop@example.com Date: 2025-08-22 Subject: We Miss You Dear حاتم زلاطيمو, We haven't seen yo..."
CUST-172-note.txt,note,1.266,Customer ID: CUST-172 Retention note (2026-02-19): Customer showed churn signals. Engagement score: ...
CUST-115-note.txt,note,1.284,Customer ID: CUST-115 Retention note (2026-02-09): Customer showed churn signals. Engagement score: ...
CUST-011-note.txt,note,1.311,Customer ID: CUST-011 Onboarding note (2025-04-19): New customer in segment 'at_risk'. First purchas...


---
## Use Case 7 — Audit Trail

**Business scenario**: Trust & Safety requires a full record of every data access — who queried what, when, and what was returned. The audit trail is immutable and written to `data/audit.jsonl`.

Every Level 1 action is logged automatically. No manual logging required.

In [8]:
# Run a query
result = ask("What is the Identity Verification status of this customer?", customer_id="CUST-005", user_id="trustsafety@shop.com")

# Show audit trail with persistent log
display_audit_trail(result.get("audit_trail", []), persistent_log_path="data/audit.jsonl")


node,intent,confidence,action,guardrail_passed,violations
orchestrator,informational,0.900000,nan,nan,nan
level1_knowledge,nan,nan,completed,True,[]


timestamp,agent_id,action,user_id
2026-03-12T12:52:16.168473+00:00,level1_knowledge,multi_query_search,manager@shop.com
2026-03-12T12:52:18.263493+00:00,orchestrator,classify_intent,manager@shop.com
2026-03-12T12:52:23.878586+00:00,level1_knowledge,multi_query_search,manager@shop.com
2026-03-12T12:52:26.490791+00:00,orchestrator,classify_intent,trustsafety@shop.com
2026-03-12T12:52:26.524140+00:00,level1_knowledge,get_identity_status,trustsafety@shop.com


---
## Use Case 8 — Output Guardrails

**Business scenario**: Before any answer reaches a user, it passes through an automatic guardrail check. This protects against:

- **PII leakage** — email addresses and phone numbers are automatically redacted from outputs
- **Forbidden content** — phrases like *"financial advice"* or *"buy shares"* are blocked

The guardrail runs silently on every response. Violations are logged to the audit trail.

In [9]:
from utils import run_guardrail_demo
run_guardrail_demo()


---
## Summary

| Use Case | What the agent does | Data source |
|---|---|---|
| **1. Customer Profile** | Returns full profile: segment, risk, products, consent | Customer DB |
| **2. Identity Verification Status** | Returns status + expiry + days remaining | Customer DB |
| **3. Email Search** | Finds relevant emails by meaning, not just keywords | Email archive (ChromaDB) |
| **4. CRM Notes** | Surfaces agent notes about escalations, retention, onboarding | Notes store (ChromaDB) |
| **5. Policy Search** | Answers policy questions with cited sources | Policy library (ChromaDB) |
| **6. Cross-Source** | Searches all collections simultaneously | All sources |
| **7. Audit Trail** | Every action logged — who, what, when | `data/audit.jsonl` |
| **8. Guardrails** | PII redacted, forbidden content blocked before output | Inline check |

---

### Key business benefits

- **Speed** — answers in seconds vs. minutes of manual lookup across systems
- **Consistency** — same policy interpretation every time, no human variation
- **Trust & Safety** — every data access is logged; PII never leaks into outputs
- **Accessibility** — plain English questions, no SQL or system training needed
- **Extensibility** — drop new documents into `data/docs/` and run `ingest_folder.py` to make them searchable immediately

---

### What comes next

Level 1 is the **read-only** layer. When you need to *act* on what you find:

- **Level 2** — run SQL analytics, segment customers, generate charts
- **Level 3** — send notifications, create cases, recommend offers
- **Level 4** — run full campaigns, track KPIs, self-correct strategy